In [0]:
df = spark.table("cpg_planning.bronze.demand_statcan_retail")



In [0]:
# Shape of the data
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
print(f"Date range:")
display(df.selectExpr("min(ref_date)", "max(ref_date)"))


In [0]:

# How many rows per NAICS code
print("Rows per NAICS code:")
display(df.groupBy("naics_code", "naics_description")
    .count()
    .orderBy("naics_code"))



In [0]:
# How many suppressed values (status = 'x')
print("Suppressed values:")
display(df.groupBy("status").count())

In [0]:
display(df.groupBy("geo").count().orderBy("geo"))


In [0]:
# Check suppressed values by province
# Some smaller provinces may be heavily suppressed
from pyspark.sql.functions import col

display(df.filter(col("status") == "x")
    .groupBy("geo")
    .count()
    .orderBy("count", ascending=False))

In [0]:
# notebooks/01_demand_sensing/02_bronze_to_silver.py

from pyspark.sql.functions import col, sum as spark_sum

TARGET_NAICS = ["445", "456", "457", "458", "455"]

MAJOR_PROVINCES = [
    "Ontario", "Quebec", "British Columbia",
    "Alberta", "Manitoba", "Saskatchewan"
]

CLEAN_STATUSES = ["A", "B", "C", "D", "E", None]

df = spark.table("cpg_planning.bronze.demand_statcan_retail")

silver = (df
    .filter(col("naics_code").isin(TARGET_NAICS))
    .filter(col("geo").isin(MAJOR_PROVINCES))
    .filter(col("status").isin(CLEAN_STATUSES) | col("status").isNull())
    .select("ref_date", "geo", "naics_code", "naics_description", "value", "status"))

# Validate before writing
print(f"Silver rows: {silver.count()}")
display(silver.groupBy("naics_code", "geo").count().orderBy("naics_code", "geo"))

In [0]:
   (silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cpg_planning.silver.demand_retail_monthly"))

# Validate
count = spark.table("cpg_planning.silver.demand_retail_monthly").count()
print(f"Silver table rows: {count}")

In [0]:
# notebooks/01_demand_sensing/04_silver_to_gold.py
# Builds gold feature table from silver.
# All features lagged to prevent leakage — when predicting month T,
# only data available through month T-1 is used.

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, lag, avg, round as spark_round
)

df = spark.table("cpg_planning.silver.demand_retail_monthly")

# Window partitioned by category + province, ordered by time
w = Window.partitionBy("naics_code", "geo").orderBy("ref_date")

features = (df
    .withColumn("lag_1m",  lag("value", 1).over(w))
    .withColumn("lag_2m",  lag("value", 2).over(w))
    .withColumn("lag_3m",  lag("value", 3).over(w))
    .withColumn("lag_6m",  lag("value", 6).over(w))
    .withColumn("lag_12m", lag("value", 12).over(w))
    .withColumn("rolling_3m_avg",
        spark_round(avg("value").over(w.rowsBetween(-3, -1)), 2))
    .withColumn("rolling_6m_avg",
        spark_round(avg("value").over(w.rowsBetween(-6, -1)), 2))
)

# Drop rows where lags are null (first 12 months per partition)
features = features.dropna(subset=["lag_12m"])

print(f"Feature rows: {features.count()}")
display(features.limit(5))

In [0]:
 # Spot check — Ontario food retail in order
# lag_1m should equal the previous row's value
display(features
    .filter((col("naics_code") == "445") & (col("geo") == "Ontario"))
    .select("ref_date", "value", "lag_1m", "lag_2m", "lag_12m", "rolling_3m_avg")
    .orderBy("ref_date")
    .limit(10))

In [0]:
from pyspark.sql.functions import col, abs as spark_abs, avg

df = spark.table("cpg_planning.gold.demand_feature_table")

display(df
    .withColumn("mape_last_month", 
        spark_abs((col("value") - col("lag_1m")) / col("value")))
    .withColumn("mape_same_month_ly", 
        spark_abs((col("value") - col("lag_12m")) / col("value")))
    .groupBy("naics_code", "naics_description")
    .agg(
        avg("mape_last_month").alias("mape_last_month"),
        avg("mape_same_month_ly").alias("mape_same_month_ly")
    )
    .orderBy("naics_code"))

In [0]:
import mlflow

category_mapes = df \
    .withColumn("mape_last_month",
        spark_abs((col("value") - col("lag_1m")) / col("value"))) \
    .withColumn("mape_same_month_ly",
        spark_abs((col("value") - col("lag_12m")) / col("value"))) \
    .groupBy("naics_code") \
    .agg(
        avg("mape_last_month").alias("mape_last_month"),
        avg("mape_same_month_ly").alias("mape_same_month_ly")
    ).collect()

with mlflow.start_run(run_name="baseline_per_category"):
    for row in category_mapes:
        mlflow.log_metric(f"{row['naics_code']}_mape_last_month", 
                         row["mape_last_month"])
        mlflow.log_metric(f"{row['naics_code']}_mape_same_month_ly", 
                         row["mape_same_month_ly"])
    print("Per-category baselines logged to MLflow")

In [0]:
import requests
import zipfile
import io
import pandas as pd

# Step 1 — get the download URL from StatCan API
# Table 18100004 is CPI
api_url = "https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/18100004/en"
response = requests.get(api_url)
print(response.status_code)
print(response.json())

# Step 2 — download and extract the zip
zip_url = response.json()["object"]
zip_response = requests.get(zip_url)

# Extract CSV from zip
z = zipfile.ZipFile(io.BytesIO(zip_response.content))
csv_filename = [f for f in z.namelist() if f.endswith(".csv") and "MetaData" not in f][0]
df = pd.read_csv(z.open(csv_filename))

# print(f"Rows: {len(df)}")
# print(f"Columns: {list(df.columns)}")
# print(df.head(3))

# What product groups exist?
print(df["Products and product groups"].unique())

# What geographies?
print(df["GEO"].unique())

In [0]:
# 09_signals_to_silver.py
# Cleans and aligns external signals to monthly grain
# Handles province broadcast for Canada-level signals

from pyspark.sql.functions import col, avg, lit
from pyspark.sql import Window

MAJOR_PROVINCES = [
    "Ontario", "Quebec", "British Columbia",
    "Alberta", "Manitoba", "Saskatchewan"
]

signals = spark.table("cpg_planning.bronze.demand_signals")

# ── CPI + GAS: already provincial, just clean ─────────────────
provincial_signals = signals.filter(col("geo") != "Canada")

# ── GOOGLE TRENDS: Canada-level → broadcast to all provinces ──
trends = signals.filter(col("geo") == "Canada")

# Cross join with provinces to replicate national signal
provinces_df = spark.createDataFrame(
    [(p,) for p in MAJOR_PROVINCES], ["geo"]
)

trends_broadcast = (trends
    .drop("geo")
    .crossJoin(provinces_df))

# ── COMBINE ───────────────────────────────────────────────────
silver_signals = provincial_signals.unionByName(trends_broadcast)

# Validate
print(f"Silver signals rows: {silver_signals.count()}")
display(silver_signals
    .groupBy("signal_name", "geo")
    .count()
    .orderBy("signal_name", "geo")
    .limit(20))

In [0]:
# 10_join_signals_to_gold.py
# Joins external signals to gold feature table as new columns
# Each signal becomes a column — wide format for ML

from pyspark.sql.functions import col, lag, lit
from pyspark.sql import Window

gold = spark.table("cpg_planning.gold.demand_feature_table")
signals = spark.table("cpg_planning.silver.demand_signals_monthly")

# ── PIVOT signals to wide format ──────────────────────────────
# Each signal_name becomes a column
signals_pivot = (signals
    .groupBy("ref_date", "geo", "naics_code")
    .pivot("signal_name")
    .avg("signal_value"))

# ── JOIN to gold ──────────────────────────────────────────────
# Left join — keep all gold rows, add signals where available
gold_with_signals = gold.join(
    signals_pivot,
    on=["ref_date", "geo", "naics_code"],
    how="left"
)

print(f"Gold rows before: {gold.count()}")
print(f"Gold rows after join: {gold_with_signals.count()}")
print(f"New columns added: {len(gold_with_signals.columns) - len(gold.columns)}")
print(f"Columns: {gold_with_signals.columns}")

In [0]:
import pandas as pd
df = spark.table("cpg_planning.gold.demand_feature_table").toPandas()
df["ref_date"] = pd.to_datetime(df["ref_date"])

# Check null counts per signal column
signal_cols = [c for c in df.columns if c.endswith("_lag1")]
print("Null counts per signal:")
for col in signal_cols:
    print(f"  {col}: {df[col].isna().sum()} nulls out of {len(df)}")

# Check date range of non-null signals
print(f"\nDate range where cpi_lag1 is not null:")
print(df[df["cpi_lag1"].notna()]["ref_date"].min())
print(df[df["cpi_lag1"].notna()]["ref_date"].max())

In [0]:
df = spark.table("cpg_planning.gold.demand_feature_table").toPandas()
signal_cols = [c for c in df.columns if c.endswith("_lag1")]
for col in signal_cols:
    nulls = df[col].isna().sum()
    print(f"{col}: {nulls} nulls")

In [0]:
display(spark.table("cpg_planning.silver.demand_signals_monthly")
    .groupBy("signal_name", "naics_code")
    .count()
    .orderBy("signal_name", "naics_code"))

In [0]:
df = spark.table("cpg_planning.gold.demand_feature_table").toPandas()
signal_cols = [c for c in df.columns if c.endswith("_lag1")]
for c in signal_cols:
    print(f"{c}: {df[c].isna().sum()} nulls")

In [0]:
import pandas as pd
import numpy as np

df = spark.table("cpg_planning.gold.demand_feature_table").toPandas()
df_clean = df.dropna(subset=["cpi_lag1", "gas_price_cents_per_litre_lag1", "lag_12m", "value"])

# Correlation of signals with target
correlations = df_clean[["value", "lag_1m", "lag_12m", 
                          "cpi_lag1", "gas_price_cents_per_litre_lag1"]].corr()
print(correlations["value"].sort_values(ascending=False))

In [0]:
df = spark.table("cpg_planning.gold.demand_feature_table").toPandas()
print(f"Total rows: {len(df)}")
print(f"Nulls in lag_12m: {df['lag_12m'].isna().sum()}")
print(f"Nulls in lag_1m: {df['lag_1m'].isna().sum()}")
print(f"Date range: {df['ref_date'].min()} to {df['ref_date'].max()}")